# BS_Agent_DingDan EDA 与清洗规则沉淀

This notebook is orchestration-only. Function code lives in `src/alg/cleaning/bs_agent_dingdan.py`.

## 0. 环境与路径配置

Configure read mode and paths. Modes: `sql_sample`, `sql_full_to_parquet`, `parquet`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from alg.cleaning.bs_agent_dingdan import (
    analyze_drug_code_consistency,
    analyze_enterprise,
    analyze_hospital_level,
    analyze_identifiers,
    analyze_numeric_desensitization,
    analyze_return_void,
    analyze_status,
    build_clean_table,
    build_clean_model_audit_v2,
    build_column_maps,
    build_drug_category_map,
    build_ownership_map,
    build_paths,
    build_quality_report,
    build_region_mapping,
    ensure_output_dirs,
    load_env,
    load_input_dataframe,
    load_yaml,
    save_basic_profile,
    save_clean_outputs,
    save_data_source_and_order_name_profiles,
    save_quality_report,
    save_v2_outputs,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

mode = "sql_sample"  # sql_sample | sql_full_to_parquet | parquet
max_rows = 5000
chunksize = 50000

paths = build_paths(PROJECT_ROOT)
ensure_output_dirs(paths)
schema = load_yaml(paths.config_path)
status_map = load_yaml(paths.status_map_path)
hospital_level_map = load_yaml(paths.hospital_level_map_path)
raw_to_alias, alias_to_raw, raw_columns = build_column_maps(schema)
sql_database_url, sql_table = load_env(paths.project_root)

print(f"project_root={paths.project_root}")
print(f"mode={mode}, max_rows={max_rows}, table={sql_table}")
print("SQL_DATABASE_URL configured:", bool(sql_database_url))

## 读取数据

CSV is only a small-sample fallback. Full local cache should be Parquet.

In [ ]:
df_raw = load_input_dataframe(
    mode=mode,
    paths=paths,
    sql_database_url=sql_database_url,
    sql_table=sql_table,
    raw_columns=raw_columns,
    max_rows=max_rows,
    chunksize=chunksize,
)
df_raw.head()

## 1. 基础规模检查

In [ ]:
basic_summary, field_counts = save_basic_profile(df_raw, alias_to_raw, paths.export_eda)
basic_summary, field_counts.head(20)

## 2. 唯一标识符检查

In [ ]:
id_report, duplicate_samples = analyze_identifiers(df_raw, alias_to_raw, paths.export_eda)
id_report, duplicate_samples.head()

## 3. 数据来源与订单名称

In [ ]:
source_top, order_name_top = save_data_source_and_order_name_profiles(df_raw, alias_to_raw, paths.export_eda)
source_top.head(), order_name_top.head()

## 4. 地区字段清洗

In [ ]:
region_map, region_conflicts = build_region_mapping(df_raw, alias_to_raw, paths.export_eda, paths.export_mappings)
region_map.head(), region_conflicts.head()

## 5. 药品编码与医保编码一致性

In [ ]:
drug_code_report = analyze_drug_code_consistency(df_raw, alias_to_raw, paths.export_eda)
drug_code_report

## 6. 数值字段脱敏影响检查

Updated policy: quantity fields share multiplier q; amount fields share multiplier m. `delivery_rate`, `arrival_rate`, `overall_arrival_rate`, quantity trends, amount trends, and order frequency trends are allowed. `price_from_amount_quantity` is forbidden: do not infer real unit price from amount / quantity.

In [ ]:
numeric_report = analyze_numeric_desensitization(df_raw, alias_to_raw, paths.export_eda)
numeric_report

## 7. 订单状态枚举与初步归类

In [ ]:
status_counts, status_review, status_unmapped = analyze_status(
    df_raw, alias_to_raw, status_map, paths.export_eda, paths.export_mappings
)
status_counts.head(), status_review.head(), status_unmapped.head()

## 8. 医疗机构等级清洗

In [ ]:
hospital_counts, hospital_review, hospital_unmapped = analyze_hospital_level(
    df_raw, alias_to_raw, hospital_level_map, paths.export_eda, paths.export_mappings
)
hospital_counts.head(), hospital_review.head(), hospital_unmapped.head()

## 9. 企业字段关系检查

In [ ]:
enterprise_report = analyze_enterprise(df_raw, alias_to_raw, paths.export_eda)
enterprise_report

## 10. 药品类别编码

In [ ]:
drug_category_counts = build_drug_category_map(df_raw, alias_to_raw, paths.export_mappings)
drug_category_counts.head()

## 11. 所有制形式编码

In [ ]:
ownership_map = build_ownership_map(df_raw, alias_to_raw, paths.export_mappings)
ownership_map.head()

## 12. 退回数量与作废数量

In [ ]:
return_void_report = analyze_return_void(df_raw, alias_to_raw, paths.export_eda)
return_void_report

## 13. 生成 clean 表

In [ ]:
df_clean = build_clean_table(
    df_raw,
    schema=schema,
    raw_to_alias=raw_to_alias,
    status_map=status_map,
    hospital_level_map=hospital_level_map,
    drug_category_counts=drug_category_counts,
)
save_clean_outputs(df_clean, paths)
df_clean.head()

## 14. 生成第二轮 clean/model/audit 样本输出

In [ ]:
df_clean_v2, df_model_sample, df_audit_sample, numeric_report_v2, profile_v2 = save_v2_outputs(df_clean, paths)
df_clean_v2.head(), df_model_sample.head(), df_audit_sample.head()

## 15. 生成 Markdown 数据质量报告

In [ ]:
quality_report = build_quality_report(
    schema=schema,
    basic=basic_summary,
    status_review=status_review,
    status_unmapped=status_unmapped,
    hospital_review=hospital_review,
    hospital_unmapped=hospital_unmapped,
)
report_path = save_quality_report(quality_report, paths.export_eda)
print(report_path)
print(quality_report[:1000])
print(profile_v2[:1000])